In [ ]:
import folium
from folium import plugins
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
from shapely.geometry import Polygon
import json

# =============================================================================
# 1. CARREGAR PERÍMETROS URBANOS DE TODOS OS MUNICÍPIOS
# =============================================================================

print("Carregando perímetros urbanos...")

perimetros = {
    'Londrina': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpLondrina/perimetro.GeoJSON"),
    'Maringá': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpMaringa/perimetro.GeoJSON"),
    'Cambé': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpCambe/perimetro.GeoJSON"),
    'Ibiporã': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpIbipora/perimetro.GeoJSON"),
    'Apucarana': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpApucarana/perimetro.GeoJSON"),
    'Arapongas': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpArapongas/perimetro.GeoJSON"),
    'Marialva': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpMarialva/perimetro.GeoJSON"),
    'Sarandi': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpSarandi/perimetro.GeoJSON"),
    'Rolândia': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpRolandia/perimetro.GeoJSON"),
    'Mandaguari': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpMandaguari/perimetro.GeoJSON"),
    'Jandaia do Sul': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpjSul/perimetro.GeoJSON"),
    'Cambira': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpCambira/perimetro.GeoJSON")
}

# Converter todos para EPSG:4326
for cidade in perimetros:
    perimetros[cidade] = perimetros[cidade].to_crs('EPSG:4326')

print(f"✓ {len(perimetros)} perímetros carregados")

# =============================================================================
# 2. BAIXAR MALHA VIÁRIA INDIVIDUALMENTE PARA CADA MUNICÍPIO
# =============================================================================

print("\nBaixando malha viária por município...")

all_edges = []

for cidade, perimetro in perimetros.items():
    print(f"\n  Processando {cidade}...")
    try:
        G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')
        gdf_edges = ox.graph_to_gdfs(G, nodes=False)
        gdf_edges['cidade'] = cidade
        all_edges.append(gdf_edges)
        print(f"    ✓ {len(gdf_edges)} segmentos viários")
    except Exception as e:
        print(f"    ✗ Erro: {e}")

# Concatenar todas as malhas viárias
gdf_all_edges = pd.concat(all_edges, ignore_index=True)

print(f"\n✓ Total de segmentos viários: {len(gdf_all_edges)}")
print("\nHierarquias encontradas:")
print(gdf_all_edges['highway'].value_counts())

# =============================================================================
# 3. CARREGAR DADOS E CONVERTER PARA GEODATAFRAME
# =============================================================================

print("\nCarregando dados de land cover...")

df_uso_lda = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_londrina/inference_results_with_positions_20250609_083252.csv', sep=';')
print(f"✓ Londrina: {len(df_uso_lda):,} patches")

df_uso_ibip = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_ibipora/inference_results_with_positions_20250609_164529.csv', sep=';')
print(f"✓ Ibiporã: {len(df_uso_ibip):,} patches")

df_uso_camb = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_cambe/inference_results_with_positions_20250609_161440.csv', sep=';')
print(f"✓ Cambé: {len(df_uso_camb):,} patches")

df_uso_apuc = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_apucarana/inference_results_with_positions_20251118_114239.csv', sep=';')
print(f"✓ Apucarana: {len(df_uso_apuc):,} patches")

df_uso_arap = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_arapongas/inference_results_with_positions_20251118_121155.csv', sep=';')
print(f"✓ Arapongas: {len(df_uso_arap):,} patches")

df_uso_cambir = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_cambira/inference_results_with_positions_20251118_122253.csv', sep=';')
print(f"✓ Cambira: {len(df_uso_cambir):,} patches")

df_uso_jand = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_jandaia/inference_results_with_positions_20251118_122910.csv', sep=';')
print(f"✓ Jandaia do Sul: {len(df_uso_jand):,} patches")

df_uso_mand = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_mandaguari/inference_results_with_positions_20251118_125143.csv', sep=';')
print(f"✓ Mandaguari: {len(df_uso_mand):,} patches")

df_uso_mari = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_marialva/inference_results_with_positions_20251118_124329.csv', sep=';')
print(f"✓ Marialva: {len(df_uso_mari):,} patches")

df_uso_rol = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_rolandia/inference_results_with_positions_20251118_130921.csv', sep=';')
print(f"✓ Rolândia: {len(df_uso_rol):,} patches")

df_uso_sar = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_sarandi/inference_results_with_positions_20251118_131430.csv', sep=';')
print(f"✓ Sarandi: {len(df_uso_sar):,} patches")

df_uso_maring = pd.read_csv('/Users/fjcosta/Documents/projects/DINO_LORA/modelos/final_models/inference_results_maringa/inference_results_with_positions_20251118_133121.csv', sep=';')
print(f"✓ Maringá: {len(df_uso_maring):,} patches")

# =============================================================================
# 4. DEFINIR CORES E NOMES
# =============================================================================

class_colors = {
    'bare': '#bf8040',
    'bush': '#009900',
    'crop': '#ffdf99',
    'grass': '#33ff33',
    'hduf': '#69000d',
    'industrial': '#e93529',
    'lduf': '#c7171c',
    'mduf': '#a10e15',
    'tree': '#003300',
    'water': '#0b9fd5'
}

class_names = {
    'hduf': 'High Density Urban Fabric',
    'mduf': 'Medium Density Urban Fabric',
    'lduf': 'Low Density Urban Fabric',
    'industrial': 'Industrial',
    'crop': 'Crop',
    'tree': 'Tree',
    'bush': 'Bush',
    'bare': 'Bare Soil',
    'grass': 'Grass',
    'water': 'Water'
}

# =============================================================================
# 5. CONVERTER PATCHES PARA GEODATAFRAME
# =============================================================================

print("\nConvertendo patches para GeoDataFrame...")

def create_patch_geometry(row):
    try:
        lat, lon = map(float, row['center'].split(','))
        half_size = 0.001
        
        polygon = Polygon([
            (lon - half_size, lat - half_size),
            (lon + half_size, lat - half_size),
            (lon + half_size, lat + half_size),
            (lon - half_size, lat + half_size),
            (lon - half_size, lat - half_size)
        ])
        
        return polygon
    except:
        return None

all_patches = []

for city_name, df_city in [('Londrina', df_uso_lda), ('Ibiporã', df_uso_ibip), ('Cambé', df_uso_camb), 
                            ('Apucarana', df_uso_apuc), ('Arapongas', df_uso_arap), ('Cambira', df_uso_cambir),
                            ('Jandaia do Sul', df_uso_jand), ('Mandaguari', df_uso_mand), ('Marialva', df_uso_mari),
                            ('Rolândia', df_uso_rol), ('Sarandi', df_uso_sar), ('Maringá', df_uso_maring)]:
    
    print(f"  Processando {city_name}...")
    
    df_city['geometry'] = df_city.apply(create_patch_geometry, axis=1)
    df_city['city'] = city_name
    df_city['color'] = df_city['predicted_class'].map(class_colors)
    df_city['class_name'] = df_city['predicted_class'].map(class_names)
    
    df_valid = df_city[df_city['geometry'].notna()].copy()
    all_patches.append(df_valid[['geometry', 'city', 'predicted_class', 'color', 'class_name']])

gdf_patches = gpd.GeoDataFrame(pd.concat(all_patches, ignore_index=True), crs='EPSG:4326')

print(f"✓ Total de patches: {len(gdf_patches):,}")

# =============================================================================
# 6. CRIAR MAPA INTERATIVO
# =============================================================================

print("\nCriando mapa interativo...")

m = folium.Map(
    location=[-23.3, -51.2],
    zoom_start=10,
    tiles='CartoDB positron'
)

# =============================================================================
# 7. ADICIONAR PATCHES COMO GEOJSON POR CLASSE
# =============================================================================

print("Adicionando patches de land cover como GeoJSON...")

for class_code in gdf_patches['predicted_class'].unique():
    subset = gdf_patches[gdf_patches['predicted_class'] == class_code]
    
    if len(subset) > 0:
        color = class_colors.get(class_code, '#808080')
        class_name = class_names.get(class_code, class_code)
        
        folium.GeoJson(
            subset,
            style_function=lambda x, c=color: {
                'fillColor': c,
                'color': c,
                'weight': 0,
                'fillOpacity': 0.7
            },
            name=class_name
        ).add_to(m)
        
        print(f"  ✓ {class_name}: {len(subset):,} patches")

# =============================================================================
# 8. ADICIONAR PERÍMETROS URBANOS
# =============================================================================

print("Adicionando perímetros urbanos...")

all_perimetros = gpd.GeoDataFrame(pd.concat(perimetros.values(), ignore_index=True))

folium.GeoJson(
    all_perimetros,
    style_function=lambda x: {
        'fillColor': 'none',
        'color': 'black',
        'weight': 2,
        'fillOpacity': 0
    }
).add_to(m)

# =============================================================================
# 9. ADICIONAR MALHA VIÁRIA HIERARQUIZADA (SEM TERTIARY, UNCLASSIFIED E SERVICE)
# =============================================================================

print("Adicionando malha viária hierarquizada...")

highway_styles = {
    'motorway': {'color': '#4B0082', 'weight': 3.5, 'opacity': 1},
    'trunk': {'color': '#6A0DAD', 'weight': 3, 'opacity': 1},
    'primary': {'color': '#8B008B', 'weight': 2.5, 'opacity': 1},
    'secondary': {'color': '#9932CC', 'weight': 2, 'opacity': 1},
    'residential': {'color': '#DDA0DD', 'weight': 0.5, 'opacity': 1}
}

for highway_type, style in highway_styles.items():
    subset = gdf_all_edges[gdf_all_edges['highway'] == highway_type]
    
    if len(subset) > 0:
        folium.GeoJson(
            subset,
            style_function=lambda x, s=style: {
                'color': s['color'],
                'weight': s['weight'],
                'opacity': s['opacity']
            }
        ).add_to(m)
        
        print(f"  ✓ {highway_type}: {len(subset)} vias")

# =============================================================================
# 10. LEGENDA SUPERIOR ESQUERDA - HIERARQUIA VIÁRIA
# =============================================================================

vias_legend_html = """
<div style="position: fixed; 
           top: 10px; left: 10px; width: 250px; height: auto; 
           background-color: white; border:2px solid grey; border-radius: 10px; z-index:9999; 
           font-size:12px; padding: 10px;">
<p style="margin:0 0 10px 0; font-weight:bold;">Urban Transportation Network Hierarchy</p>
<table style="font-size:11px; border-collapse: collapse; width:100%;">
"""

for highway_type, style in highway_styles.items():
    subset = gdf_all_edges[gdf_all_edges['highway'] == highway_type]
    if len(subset) > 0:
        vias_legend_html += f"""
        <tr>
            <td style="width:30px; height:3px; background-color:{style['color']}; border:1px solid #ccc;"></td>
            <td style="padding-left:8px; vertical-align:middle;">{highway_type.capitalize()}</td>
        </tr>
        """

vias_legend_html += """
</table>
</div>
"""

m.get_root().html.add_child(folium.Element(vias_legend_html))

# =============================================================================
# 11. LEGENDA SUPERIOR DIREITA - LAND COVER CLASSES
# =============================================================================

landcover_legend_html = """
<div style="position: fixed; 
           top: 10px; right: 10px; width: 280px; height: auto; 
           background-color: white; border:2px solid grey; border-radius: 10px; z-index:9999; 
           font-size:12px; padding: 10px;">
<p style="margin:0 0 10px 0; font-weight:bold;">Land Cover Classification</p>
<table style="font-size:11px; border-collapse: collapse; width:100%;">
"""

for class_code, class_name in class_names.items():
    if class_code in class_colors:
        color = class_colors[class_code]
        landcover_legend_html += f"""
        <tr>
            <td style="width:20px; height:20px; background-color:{color}; border:1px solid #ccc;"></td>
            <td style="padding-left:8px; vertical-align:middle;">{class_name}</td>
        </tr>
        """

landcover_legend_html += """
</table>
</div>
"""

m.get_root().html.add_child(folium.Element(landcover_legend_html))

# =============================================================================
# 12. SALVAR MAPA
# =============================================================================

output_path = '/Users/fjcosta/Documents/landCoverlandValue/landcover_transport/LandCover_DINO_TransportStructure.html'
m.save(output_path)

print(f"\n✓ Mapa salvo em: {output_path}")

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✓ Tamanho do arquivo: {file_size_mb:.2f} MB")
print("\nAbra o arquivo HTML no navegador!")

Carregando perímetros urbanos...
✓ 12 perímetros carregados

Baixando malha viária por município...

  Processando Londrina...


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_10487/3899657836.py:48: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')


    ✓ 35101 segmentos viários

  Processando Maringá...


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_10487/3899657836.py:48: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')


    ✓ 26877 segmentos viários

  Processando Cambé...


...

    ✓ 2694 segmentos viários

  Processando Cambira...
    ✓ 916 segmentos viários

✓ Total de segmentos viários: 128319

Hierarquias encontradas:
highway
residential                     101739
tertiary                         10767
secondary                         5961
unclassified                      3831
primary                           1760
trunk                             1180
living_street                      915
secondary_link                     463
tertiary_link                      424
trunk_link                         368
motorway                           297
motorway_link                      264
primary_link                       209
[unclassified, residential]         66
busway                              44
[tertiary, residential]             13
[tertiary, unclassified]             6
[living_street, residential]         4
[motorway_link, motorway]            2
[tertiary, motorway_link]            2
[secondary, motorway_link]           1
[motorway, motorway_link] 

/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_10487/3899657836.py:48: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')


✓ Londrina: 18,302 patches
✓ Ibiporã: 3,903 patches
✓ Cambé: 5,905 patches
✓ Apucarana: 9,043 patches
✓ Arapongas: 10,671 patches
✓ Cambira: 238 patches
✓ Jandaia do Sul: 1,693 patches
✓ Mandaguari: 3,520 patches
✓ Marialva: 2,297 patches
✓ Rolândia: 3,771 patches
✓ Sarandi: 2,342 patches
✓ Maringá: 11,481 patches

Convertendo patches para GeoDataFrame...
  Processando Londrina...
  Processando Ibiporã...
  Processando Cambé...
  Processando Apucarana...
  Processando Arapongas...
  Processando Cambira...
  Processando Jandaia do Sul...
  Processando Mandaguari...
  Processando Marialva...
  Processando Rolândia...
  Processando Sarandi...
  Processando Maringá...
✓ Total de patches: 73,166

Criando mapa interativo...
Adicionando patches de land cover como GeoJSON...
  ✓ Crop: 27,487 patches
  ✓ Tree: 5,644 patches
  ✓ Bush: 2,994 patches
  ✓ Medium Density Urban Fabric: 3,599 patches
  ✓ Low Density Urban Fabric: 20,078 patches
  ✓ Bare Soil: 2,080 patches
  ✓ Grass: 3,217 patches
  ✓

/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_10487/3899657836.py:223: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_perimetros = gpd.GeoDataFrame(pd.concat(perimetros.values(), ignore_index=True))


  ✓ primary: 1760 vias
  ✓ secondary: 5961 vias
  ✓ residential: 101739 vias

✓ Mapa salvo em: /Users/fjcosta/Documents/landCoverlandValue/landvaue_transport/LandValue_TabPFN_TransportStructure.html
✓ Tamanho do arquivo: 92.37 MB

Abra o arquivo HTML no navegador!
